### Day3 - Part 4. 종합 실습: PyTorch Transformer로 NSMC 한국어 영화 리뷰 감성 분석하기

이번 실습에서는 PyTorch에서 제공하는 내장 Transformer 모듈을 사용하여 한국어 영화 리뷰 감성 분석을 수행해보겠습니다.

**PyTorch 내장 Transformer 모듈의 장점:**
- 최적화된 성능과 안정성
- 복잡한 구현 없이 간단한 API로 사용 가능
- 다양한 하이퍼파라미터 조정 옵션

**PyTorch nn.Transformer 모듈 사용 예시:**
```python
torch_transformer = nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    batch_first=True  # 이 옵션을 True로 주면 (Batch, Seq, Dim) 순서로 입력을 받습니다.
)
```

**NSMC 데이터셋:**
- 네이버 영화 리뷰 감성 분석 데이터셋
- 20만 개의 한국어 영화 리뷰 (긍정/부정 라벨)
- 실제 한국어 텍스트 처리 경험 제공

#### 1. 필요한 라이브러리 및 데이터 로드


In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import re
from collections import Counter
import plotly.express as px
from tqdm import tqdm
import math
from tqdm import tqdm
tqdm.pandas()

# KiwiPiePy 설치가 필요한 경우
# !pip install kiwipiepy
from kiwipiepy import Kiwi

# 한국어 형태소 분석기 초기화
kiwi = Kiwi()

print("라이브러리 로드 완료!")


라이브러리 로드 완료!


In [4]:
# NSMC 데이터 로드
train_data = pd.read_csv('../../datasets/text/nsmc/ratings_train.txt', sep='\t')
test_data = pd.read_csv('../../datasets/text/nsmc/ratings_test.txt', sep='\t')

# 결측값 제거
train_data = train_data.dropna()
test_data = test_data.dropna()

print(f"훈련 데이터 크기: {len(train_data)}")
print(f"테스트 데이터 크기: {len(test_data)}")

훈련 데이터 크기: 149995
테스트 데이터 크기: 49997


In [5]:
print("데이터 샘플:")
train_data.head()

데이터 샘플:


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
# 라벨 분포 확인
train_data.label.value_counts()

label
0    75170
1    74825
Name: count, dtype: int64

#### 2. 한국어 텍스트 전처리

한국어는 영어와 달리 형태소 분석이 필요합니다. KiwiPiePy를 사용하여 의미있는 형태소만 추출하겠습니다.


In [9]:
def preprocess_korean_text(text):
    """
    한국어 텍스트 전처리 및 형태소 분석
    
    Args:
        text (str): 원본 텍스트
    
    Returns:
        list: 전처리된 토큰 리스트
    """
    # HTML 태그 및 특수문자 제거
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)
    text = text.strip()
    
    if not text:
        return []
    
    # Kiwi를 사용한 형태소 분석
    try:
        tokens = kiwi.analyze(text)
        # 명사, 동사, 형용사, 부사 등 의미있는 형태소만 추출
        meaningful_tokens = []
        for token in tokens[0][0]:
            if token.tag in ['NNG', 'NNP', 'VV', 'VA', 'MAG', 'XR'] and len(token.form) > 1:
                meaningful_tokens.append(token.form)
        return meaningful_tokens
    except:
        # 형태소 분석 실패 시 단순 공백 분리
        return text.split()

# 샘플 데이터로 전처리 테스트 (시간 단축을 위해)
sample_size = 50000  # 전체 데이터 사용 시 시간이 오래 걸릴 수 있음
train_sample = train_data.sample(n=sample_size, random_state=42)

print("텍스트 전처리 중...")
train_sample['processed_text'] = train_sample['document'].progress_apply(preprocess_korean_text)

텍스트 전처리 중...


  0%|          | 0/50000 [00:00<?, ?it/s]

100%|██████████| 50000/50000 [00:25<00:00, 1994.26it/s]


In [10]:

# 빈 리스트 제거
train_sample = train_sample[train_sample['processed_text'].apply(len) > 0]

print(f"전처리 완료! 유효한 데이터 수: {len(train_sample)}")
print("전처리 예시:")
for i in range(3):
    print(f"원본: {train_sample.iloc[i]['document']}")
    print(f"전처리: {train_sample.iloc[i]['processed_text']}")
    print(f"라벨: {train_sample.iloc[i]['label']}")
    print("-" * 50)


전처리 완료! 유효한 데이터 수: 47892
전처리 예시:
원본: 원본이 최고
전처리: ['원본', '최고']
라벨: 1
--------------------------------------------------
원본: 스릴감과 훈훈함이 있는 영화.
전처리: ['스릴', '훈훈', '영화']
라벨: 1
--------------------------------------------------
원본: 굉장히 저평가되는 영화중 하나라고 생각함
전처리: ['굉장히', '평가', '영화', '생각']
라벨: 1
--------------------------------------------------


#### 3. 단어 사전 구축 및 정수 인코딩


In [11]:
# 모든 토큰 수집
all_tokens = [token for tokens in train_sample['processed_text'] for token in tokens]
token_counts = Counter(all_tokens)

# 상위 빈도 단어로 어휘 사전 구성
vocab_size = 10000
vocab = [word for word, count in token_counts.most_common(vocab_size - 3)]  # 특수 토큰 3개 제외

# 특수 토큰 추가
word_to_idx = {
    '<pad>': 0,  # 패딩
    '<unk>': 1,  # 미지의 단어
    '<cls>': 2   # 분류 토큰 (Transformer에서 사용)
}

# 일반 단어들 추가
for idx, word in enumerate(vocab):
    word_to_idx[word] = idx + 3

print(f"어휘 사전 크기: {len(word_to_idx)}")
print(f"가장 빈번한 단어 10개: {vocab[:10]}")

어휘 사전 크기: 10000
가장 빈번한 단어 10개: ['영화', '너무', '정말', '재밌', '진짜', '연기', '만들', '나오', '최고', '평점']


In [12]:
def encode_text(tokens, word_to_idx, max_len=128):
    """
    토큰을 정수로 인코딩하고 패딩 적용
    
    Args:
        tokens (list): 토큰 리스트
        word_to_idx (dict): 단어-인덱스 매핑
        max_len (int): 최대 시퀀스 길이
    
    Returns:
        list: 인코딩된 시퀀스
    """
    # CLS 토큰으로 시작
    encoded = [word_to_idx['<cls>']]
    
    # 토큰 인코딩 (최대 길이 -1 까지, CLS 토큰 때문에)
    for token in tokens[:max_len-1]:
        encoded.append(word_to_idx.get(token, word_to_idx['<unk>']))
    
    # 패딩
    while len(encoded) < max_len:
        encoded.append(word_to_idx['<pad>'])
    
    return encoded

In [14]:
# 시퀀스 길이 분석
sequence_lengths = [len(tokens) for tokens in train_sample['processed_text']]

print(f"평균 시퀀스 길이: {np.mean(sequence_lengths):.2f}")
print(f"중앙값 시퀀스 길이: {np.median(sequence_lengths):.2f}")
print(f"최대 시퀀스 길이: {np.max(sequence_lengths)}")
print(f"최소 시퀀스 길이: {np.min(sequence_lengths)}")

# 시퀀스 길이 분포 시각화
fig = px.histogram(
    x=sequence_lengths,
    nbins=50,
    title='텍스트 시퀀스 길이 분포',
    labels={'x': '시퀀스 길이', 'y': '빈도'},
    color_discrete_sequence=['lightblue']
)

fig.update_layout(showlegend=False)
fig.show()

# 백분위수 확인
percentiles = [50, 75, 90, 95, 99]
for p in percentiles:
    length = np.percentile(sequence_lengths, p)
    print(f"{p}번째 백분위수: {length:.0f}")

# max_len 설정을 위한 권장사항
print(f"\nmax_len 설정 권장사항:")
print(f"- 75% 커버리지: {np.percentile(sequence_lengths, 75):.0f}")
print(f"- 90% 커버리지: {np.percentile(sequence_lengths, 90):.0f}")
print(f"- 95% 커버리지: {np.percentile(sequence_lengths, 95):.0f}")


평균 시퀀스 길이: 5.77
중앙값 시퀀스 길이: 4.00
최대 시퀀스 길이: 37
최소 시퀀스 길이: 1


50번째 백분위수: 4
75번째 백분위수: 7
90번째 백분위수: 12
95번째 백분위수: 17
99번째 백분위수: 24

max_len 설정 권장사항:
- 75% 커버리지: 7
- 90% 커버리지: 12
- 95% 커버리지: 17


In [15]:

# 텍스트 인코딩
max_len = 12
encoded_texts = [encode_text(tokens, word_to_idx, max_len) for tokens in train_sample['processed_text']]

print(f"인코딩된 시퀀스 길이: {len(encoded_texts[0])}")
print(f"첫 번째 시퀀스 예시: {encoded_texts[0][:10]}...")


인코딩된 시퀀스 길이: 12
첫 번째 시퀀스 예시: [2, 5261, 11, 0, 0, 0, 0, 0, 0, 0]...


#### 4. 데이터 분할 및 DataLoader 생성


In [16]:
# 데이터 분할
X = np.array(encoded_texts)
y = train_sample['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 텐서 변환
X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.LongTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val)

# DataLoader 생성
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"훈련 데이터: {len(X_train)}")
print(f"검증 데이터: {len(X_val)}")
print(f"배치 수 (훈련): {len(train_loader)}")


훈련 데이터: 38313
검증 데이터: 9579
배치 수 (훈련): 1198


#### 5. PyTorch 내장 Transformer를 사용한 감성 분석 모델

**PyTorch nn.Transformer 모듈 주요 파라미터:**
- `d_model`: 모델의 차원 (임베딩 차원과 동일해야 함)
- `nhead`: 멀티헤드 어텐션의 헤드 수
- `num_encoder_layers`: 인코더 레이어 수
- `num_decoder_layers`: 디코더 레이어 수 (분류 태스크에서는 사용하지 않음)
- `dim_feedforward`: 피드포워드 네트워크의 차원
- `dropout`: 드롭아웃 비율
- `batch_first`: 배치 차원을 첫 번째로 할지 여부


In [19]:
class PositionalEncoding(nn.Module):
    """
    위치 인코딩 클래스
    Transformer는 순서 정보가 없으므로 위치 정보를 추가해야 함
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [17]:
class TransformerSentimentClassifier(nn.Module):
    """
    PyTorch 내장 Transformer를 사용한 감성 분류 모델
    """
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, 
                 dim_feedforward=2048, max_len=128, dropout=0.1):
        super(TransformerSentimentClassifier, self).__init__()
        
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len) # 순서 정보 추가
        
        # PyTorch 내장 TransformerEncoder 사용
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)
        
        # 임베딩 초기화
        self.embedding.weight.data.uniform_(-0.1, 0.1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        
        # 패딩 마스크 생성 (패딩 토큰은 0)
        padding_mask = (x == 0)  # (batch_size, seq_len)
        
        # 임베딩 + 위치 인코딩
        embedded = self.embedding(x) * math.sqrt(self.d_model)  # 스케일링
        embedded = embedded.transpose(0, 1)  # (seq_len, batch_size, d_model)
        embedded = self.pos_encoding(embedded)
        embedded = embedded.transpose(0, 1)  # (batch_size, seq_len, d_model)
        
        # Transformer 인코더 통과
        transformer_output = self.transformer_encoder(
            embedded, 
            src_key_padding_mask=padding_mask # 패딩 토큰은 무시됨
        )
        
        # CLS 토큰 (첫 번째 토큰) 사용하여 분류
        cls_output = transformer_output[:, 0, :]  # (batch_size, d_model)
        
        # 분류 레이어
        output = self.dropout(cls_output)
        logits = self.classifier(output).squeeze(-1)  # (batch_size,)
        
        return logits


In [20]:
# 모델 하이퍼파라미터
d_model = 256  # 실습용으로 작게 설정
nhead = 8
num_layers = 4
dim_feedforward = 1024

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

model = TransformerSentimentClassifier(
    vocab_size=len(word_to_idx),
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    max_len=max_len
).to(device)

# 모델 파라미터 수 계산
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'모델 파라미터 수: {count_parameters(model):,}')
print("모델 구조:")
print(model)


사용 디바이스: cpu
모델 파라미터 수: 5,719,297
모델 구조:
TransformerSentimentClassifier(
  (embedding): Embedding(10000, 256, padding_idx=0)
  (pos_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)


#### 6. 모델 학습


In [22]:
# 손실 함수 및 옵티마이저
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# 학습률 스케줄러 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # 그래디언트 클리핑
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            # 정확도 계산
            preds = torch.round(torch.sigmoid(outputs))
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total
    return total_loss / len(loader), accuracy

In [23]:
# 모델 학습
num_epochs = 5
best_val_acc = 0
train_losses = []
val_losses = []
val_accuracies = []

print("모델 학습 시작...")
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # 학습률 스케줄러 업데이트
    scheduler.step(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # 최고 성능 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_transformer_model.pth')
        print(f"새로운 최고 성능! 모델 저장됨 (Acc: {val_acc:.4f})")

print(f"학습 완료! 최고 검증 정확도: {best_val_acc:.4f}")


모델 학습 시작...

Epoch 1/5


Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:505: UserWarning:

The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)

Evaluating: 100%|██████████| 300/300 [00:09<00:00, 31.60it/s]


Train Loss: 0.5263
Val Loss: 0.4920, Val Acc: 0.7531
Learning Rate: 0.000100
새로운 최고 성능! 모델 저장됨 (Acc: 0.7531)

Epoch 2/5


Evaluating: 100%|██████████| 300/300 [00:06<00:00, 48.01it/s]


Train Loss: 0.4243
Val Loss: 0.4625, Val Acc: 0.7842
Learning Rate: 0.000100
새로운 최고 성능! 모델 저장됨 (Acc: 0.7842)

Epoch 3/5


...

Train Loss: 0.3686
Val Loss: 0.4620, Val Acc: 0.7911
Learning Rate: 0.000100
새로운 최고 성능! 모델 저장됨 (Acc: 0.7911)

Epoch 4/5


Evaluating: 100%|██████████| 300/300 [00:07<00:00, 42.84it/s]


Train Loss: 0.3185
Val Loss: 0.5115, Val Acc: 0.7815
Learning Rate: 0.000100

Epoch 5/5


Evaluating: 100%|██████████| 300/300 [00:07<00:00, 42.54it/s]

Train Loss: 0.2706
Val Loss: 0.6320, Val Acc: 0.7754
Learning Rate: 0.000100
학습 완료! 최고 검증 정확도: 0.7911


#### 7. 학습 결과 시각화


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 학습 곡선 시각화
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Loss', 'Validation Accuracy'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}]]
)

epochs = list(range(1, len(train_losses) + 1))

# Loss 그래프
fig.add_trace(
    go.Scatter(x=epochs, y=train_losses, name='Train Loss', line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=epochs, y=val_losses, name='Val Loss', line=dict(color='red')),
    row=1, col=1
)

# Accuracy 그래프
fig.add_trace(
    go.Scatter(x=epochs, y=val_accuracies, name='Val Accuracy', line=dict(color='green')),
    row=1, col=2
)

fig.update_layout(title='Transformer 모델 학습 곡선', height=400)
fig.show()

print(f"최종 검증 정확도: {val_accuracies[-1]:.4f}")
print(f"최고 검증 정확도: {max(val_accuracies):.4f}")


#### 8. 새로운 한국어 리뷰로 예측하기


In [ ]:
def predict_sentiment(text, model, word_to_idx, device, max_len=128):
    """
    새로운 텍스트의 감성을 예측하는 함수
    
    Args:
        text (str): 예측할 텍스트
        model: 학습된 모델
        word_to_idx (dict): 단어-인덱스 매핑
        device: 연산 디바이스
        max_len (int): 최대 시퀀스 길이
    
    Returns:
        tuple: (예측 결과, 확률)
    """
    model.eval()
    
    # 텍스트 전처리
    tokens = preprocess_korean_text(text)
    if not tokens:
        return "예측 불가", 0.5
    
    # 인코딩
    encoded = encode_text(tokens, word_to_idx, max_len)
    input_tensor = torch.LongTensor([encoded]).to(device)
    
    # 예측
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.sigmoid(output).item()
    
    sentiment = "긍정" if prob > 0.5 else "부정"
    return sentiment, prob

# 최고 성능 모델 로드
model.load_state_dict(torch.load('best_transformer_model.pth'))

# 테스트 리뷰들
test_reviews = [
    "정말 재미있는 영화였어요! 시간 가는 줄 몰랐습니다.",
    "완전 최악의 영화네요. 시간 아까웠어요.",
    "배우들의 연기가 너무 자연스럽고 스토리도 감동적이었습니다.",
    "지루하고 재미없어서 중간에 나왔습니다.",
    "볼만한 영화예요. 그런데 조금 아쉬운 부분도 있네요."
]

print("=== 새로운 리뷰 감성 예측 ===")
for i, review in enumerate(test_reviews, 1):
    sentiment, prob = predict_sentiment(review, model, word_to_idx, device, max_len)
    print(f"\n{i}. 리뷰: {review}")
    print(f"   예측: {sentiment} (확률: {prob:.4f})")
    
    # 신뢰도 표시
    confidence = abs(prob - 0.5) * 2
    print(f"   신뢰도: {confidence:.4f}")


#### 9. 과제 및 추가 실험

**기본 과제:**
1. 다른 하이퍼파라미터 조합으로 실험해보기
   - `d_model`: 128, 256, 512
   - `nhead`: 4, 8, 16
   - `num_layers`: 2, 4, 6

2. 다른 풀링 방법 시도하기
   - 평균 풀링 (mean pooling)
   - 최대 풀링 (max pooling)
   - 가중 평균 풀링

**심화 과제:**
1. 사전 훈련된 한국어 임베딩 사용하기
2. 데이터 증강 기법 적용하기
3. 어텐션 가중치 시각화하기
4. 다른 한국어 데이터셋으로 전이 학습하기

**성능 비교:**
- LSTM vs Transformer 성능 비교
- 학습 시간 및 메모리 사용량 비교
- 다양한 길이의 텍스트에 대한 성능 분석

**PyTorch 내장 모듈 활용법:**

1. **nn.TransformerEncoder 사용법:**
```python
encoder_layer = nn.TransformerEncoderLayer(
    d_model=512, nhead=8, dim_feedforward=2048, dropout=0.1, batch_first=True
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)
```

2. **nn.Transformer 전체 모듈 사용법:**
```python
transformer = nn.Transformer(
    d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=0,
    dim_feedforward=2048, dropout=0.1, batch_first=True
)
```

3. **주요 차이점:**
   - `TransformerEncoder`는 인코더만 사용 (분류 태스크에 적합)
   - `Transformer`는 인코더+디코더 (seq2seq 태스크에 적합)
   - 분류 태스크에서는 `num_decoder_layers=0`으로 설정하여 인코더만 사용


In [ ]:
# ============================================================================
# 과제 1: 평균 풀링을 사용한 Transformer 모델 구현
# ============================================================================

class TransformerWithMeanPooling(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, 
                 dim_feedforward=1024, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # 과제 1-1: 평균 풀링 구현하기
        # 패딩 마스크를 고려하여 실제 토큰들만의 평균을 계산하세요
        padding_mask = (x == 0)
        
        embedded = self.embedding(x) * math.sqrt(self.d_model)
        embedded = embedded.transpose(0, 1)
        embedded = self.pos_encoding(embedded)
        embedded = embedded.transpose(0, 1)
        
        transformer_output = self.transformer_encoder(embedded, src_key_padding_mask=padding_mask)
        
        # 평균 풀링 구현
        mask = (~padding_mask).unsqueeze(-1).float()  # (batch_size, seq_len, 1)
        masked_output = transformer_output * mask
        pooled_output = masked_output.sum(dim=1) / mask.sum(dim=1)  # (batch_size, d_model)
        
        output = self.dropout(pooled_output)
        logits = self.classifier(output).squeeze(-1)
        
        return logits

# ============================================================================
# 과제 2: 하이퍼파라미터 실험 및 성능 비교
# ============================================================================

def experiment_hyperparameters():
    """
    과제 2-1: 다양한 하이퍼파라미터 조합으로 실험하기
    """
    configs = [
        {'d_model': 128, 'nhead': 4, 'num_layers': 2},
        {'d_model': 256, 'nhead': 8, 'num_layers': 4},
        {'d_model': 512, 'nhead': 8, 'num_layers': 6},
    ]
    
    results = []
    for config in configs:
        print(f"실험 중: {config}")
        
        # 모델 생성
        model_exp = TransformerSentimentClassifier(
            vocab_size=len(word_to_idx),
            d_model=config['d_model'],
            nhead=config['nhead'],
            num_layers=config['num_layers'],
            dim_feedforward=config['d_model'] * 4,
            max_len=max_len
        ).to(device)
        
        # 과제 2-2: 각 설정으로 모델 학습 및 평가 구현
        # TODO: 학습 루프 구현
        # TODO: 검증 데이터로 성능 평가
        # TODO: 결과 저장
        
        print(f"모델 파라미터 수: {count_parameters(model_exp):,}")
    
    return results

# ============================================================================
# 과제 3: 어텐션 가중치 시각화
# ============================================================================

def extract_attention_weights(model, input_text):
    """
    과제 3-1: 어텐션 가중치를 추출하는 함수 구현
    """
    model.eval()
    
    # 텍스트 전처리 및 인코딩
    tokens = preprocess_korean_text(input_text)
    encoded = encode_text(tokens, word_to_idx, max_len)
    input_tensor = torch.LongTensor([encoded]).to(device)
    
    # 어텐션 가중치를 저장할 리스트
    attention_weights = []
    
    # 훅 함수 정의
    def hook_fn(module, input, output):
        # TransformerEncoderLayer의 self-attention 가중치 추출
        if hasattr(module, 'self_attn'):
            attention_weights.append(output[1])  # attention weights
    
    # 훅 등록
    hooks = []
    for layer in model.transformer_encoder.layers:
        hook = layer.register_forward_hook(hook_fn)
        hooks.append(hook)
    
    # 순전파 실행
    with torch.no_grad():
        _ = model(input_tensor)
    
    # 훅 제거
    for hook in hooks:
        hook.remove()
    
    return attention_weights, tokens

def visualize_attention_weights(attention_weights, tokens, layer_idx=0, head_idx=0):
    """
    과제 3-2: 어텐션 가중치를 히트맵으로 시각화하기
    """
    # TODO: plotly를 사용하여 어텐션 히트맵 생성
    # 힌트: go.Heatmap 사용
    pass

# ============================================================================
# 과제 4: 최대 풀링 모델 구현
# ============================================================================

class TransformerWithMaxPooling(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, 
                 dim_feedforward=1024, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # 과제 4-1: 최대 풀링 구현하기
        # 패딩을 제외한 토큰들 중에서 최대값을 선택하세요
        padding_mask = (x == 0)
        
        embedded = self.embedding(x) * math.sqrt(self.d_model)
        embedded = embedded.transpose(0, 1)
        embedded = self.pos_encoding(embedded)
        embedded = embedded.transpose(0, 1)
        
        transformer_output = self.transformer_encoder(embedded, src_key_padding_mask=padding_mask)
        
        # 최대 풀링 구현
        mask = (~padding_mask).unsqueeze(-1).float()  # (batch_size, seq_len, 1)
        masked_output = transformer_output * mask
        
        # 패딩 위치를 매우 작은 값으로 설정하여 최대값 선택에서 제외
        masked_output = masked_output + (padding_mask.unsqueeze(-1).float() * -1e9)
        pooled_output = torch.max(masked_output, dim=1)[0]  # (batch_size, d_model)
        
        output = self.dropout(pooled_output)
        logits = self.classifier(output).squeeze(-1)
        
        return logits

# ============================================================================
# 과제 실행 및 테스트
# ============================================================================

print("=== Transformer 심화 과제 시작 ===")
print("과제 1: 평균 풀링 모델")
print("과제 2: 하이퍼파라미터 실험")
print("과제 3: 어텐션 가중치 시각화")
print("과제 4: 최대 풀링 모델")

# 과제 1 테스트
print("\n=== 과제 1: 평균 풀링 모델 테스트 ===")
mean_pooling_model = TransformerWithMeanPooling(
    vocab_size=len(word_to_idx),
    d_model=128,
    nhead=4,
    num_layers=2,
    max_len=max_len
).to(device)

print(f"평균 풀링 모델 파라미터 수: {count_parameters(mean_pooling_model):,}")

# 과제 4 테스트
print("\n=== 과제 4: 최대 풀링 모델 테스트 ===")
max_pooling_model = TransformerWithMaxPooling(
    vocab_size=len(word_to_idx),
    d_model=128,
    nhead=4,
    num_layers=2,
    max_len=max_len
).to(device)

print(f"최대 풀링 모델 파라미터 수: {count_parameters(max_pooling_model):,}")

# 간단한 테스트
test_input = torch.randint(1, 100, (2, max_len)).to(device)
test_output = mean_pooling_model(test_input)
print(f"테스트 출력 크기: {test_output.shape}")
print("평균 풀링 모델이 정상적으로 동작합니다!")
